<a href="https://colab.research.google.com/github/Ahella-Bassem-Mohammed/Cat-Dog-App/blob/main/cat_dog_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models
import os
from google.colab import files

In [8]:
path = kagglehub.dataset_download("marquis03/cats-and-dogs")
train_dir = os.path.join(path, 'train')
val_dir = os.path.join(path, 'test')


Using Colab cache for faster access to the 'cats-and-dogs' dataset.


In [9]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 275 files belonging to 2 classes.
Using 220 files for training.
Found 275 files belonging to 2 classes.
Using 55 files for validation.


In [10]:

base_model = tf.keras.applications.MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(160, 160, 3))
x = layers.Rescaling(1./255)(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model_transfer = tf.keras.Model(inputs, outputs)



In [11]:
model_cnn = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [12]:
results = {}

for name, model in [("MobileNet", model_transfer), ("Custom_CNN", model_cnn)]:
    print(f"\n--- Training {name} ---")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(train_ds, validation_data=val_ds, epochs=2)


    acc = history.history['val_accuracy'][-1]
    results[name] = acc
    print(f"\n✅ {name} Validation Accuracy: {acc*100:.2f}%")
    model.save(f'{name.lower()}.keras')


--- Training MobileNet ---
Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5322 - loss: 0.7468 - val_accuracy: 0.6182 - val_loss: 0.5882
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 826ms/step - accuracy: 0.8198 - loss: 0.4710 - val_accuracy: 0.8727 - val_loss: 0.3660

✅ MobileNet Validation Accuracy: 87.27%

--- Training Custom_CNN ---
Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.4861 - loss: 1.6907 - val_accuracy: 0.6182 - val_loss: 0.6803
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.6406 - loss: 0.6591 - val_accuracy: 0.6182 - val_loss: 0.6625

✅ Custom_CNN Validation Accuracy: 61.82%


In [13]:
print("\n--- Final Comparison ---")
for name, acc in results.items():
    print(f"{name}: {acc*100:.2f}%")


--- Final Comparison ---
MobileNet: 87.27%
Custom_CNN: 61.82%


In [15]:
files.download('mobilenet.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>